# Ausloans Lender Decision Engine

In [2]:
import re
import numpy as np
import pandas as pd
import json
from datetime import date

import warnings
warnings.filterwarnings('ignore')

### Step 1: Read Applicant dataset

In [3]:
# Read Loan Application Dataset
df_data = pd.read_csv('loan_application_dataset.csv')

# Applicant Data Transformation
df_data.replace({'N': 0, 'Y': 1}, inplace=True)
df_data.replace(-991, 0, inplace=True)  # monthExBankrupt
df_data.replace('-991', 0, inplace=True)  # dont delete, some come as string
df_data.replace(-993, 0, inplace=True)  # dont delete, some come as string
df_data.replace('-993', 0, inplace=True)  # dont delete, some come as string
df_data.replace('X', 7, inplace=True)  # CCR
df_data.replace(-903, 0, inplace=True)
df_data.replace('-903', 0, inplace=True)
df_data.fillna(0, inplace=True)

# Identify and drop duplicate columns
duplicate_columns = df_data.columns[df_data.columns.duplicated()]
df_data = df_data.drop(columns=duplicate_columns)

# Drop duplicate records by loanApplicationId
df_data.drop_duplicates(subset=['loanApplicationId'], keep='first', inplace=True)

# Lowercase all values
df_loan_app = df_data.applymap(lambda x: x.lower() if isinstance(x, str) else x)

# Select One Applicant
df_loan_app = df_loan_app[(df_loan_app['loanApplicationId'] == 738380)]
df_loan_app

,loanApplicationId,applicantId,gender,applicantAge,maritalStatus,isLivingTogether,dependents,visaType,currentResidenceStatus,currentResidencePostcode,...,totalRentalIncomeMonthly,totalLivingExpensesMonthly,totalChildExpensesMonthly,totalGamblingExpensesMonthly,totalLiabilitiesMonthly,totalAssetsMonthly,totalDependents,totalAdditionalIncome,totalSpouseIncome,totalSavings
4294,738380,552822,male,39,married,True,2,citizen,mortgaged,7307,...,0.0,2569.28,1352.67,0.0,3731.04,0.0,2,0.0,4939.62,30000.0


### Step 2: Read Rules dataset

In [4]:
# Read Product Matrix Dataset 
df_product_matrix = pd.read_csv('product_matrix_rule_dataset.csv')
df_product_matrix.head(2)

,ProductId,LenderId,ProductName,ProductGradeName,Enabled,MinLoan,MaxLoan,MinTerm,MaxTerm,AssetCondition,...,RuleSACC,RuleBankStatement,MaxFinancialDishonorAllow,MaxNonFinancialDishonorAllow,MaxGamblingExpByMonth,MaxGamblingByIncomePercent,CentrelinkAcceptedPercent,CentrelinkRelativeToIncome,CentrelinkAcceptedType,MitigatingFactor
0,9001,101,Pepper Finance - Consumer A - Car,A,True,5000,150000,12,84,NEW,...,STATEMENT_REQ=TRUE; IF(PAYDAY_ENQUIRY>0 & ENQU...,STATEMENT_REQ=TRUE; IF(PAYDAY_ENQUIRY>0 & ENQU...,7,99,NaN,0.1,1.0,NaN,NO_NEW_START,ASSET_BACKED=FALSE;IF(SAVINGS>=$50000)
1,9002,101,Pepper Finance - Consumer A - Car,A,True,5000,150000,12,84,USED,...,STATEMENT_REQ=TRUE; IF(PAYDAY_ENQUIRY>0 & ENQU...,STATEMENT_REQ=TRUE; IF(PAYDAY_ENQUIRY>0 & ENQU...,7,99,NaN,0.1,1.0,NaN,NO_NEW_START,ASSET_BACKED=FALSE;IF(SAVINGS>=$50000)


In [5]:
# ****************************
#    Lender Rules 
# ****************************
# Rule 1: Check whether product is enabled
def is_product_enabled(rule_data: dict) -> bool:
    # return:: True--> if product is enabled
    try:
        return bool(rule_data.get('isEnable', True))
    except KeyError as e:
        return False

# Rule 2: Min Max Loan Amount
def is_loan_amount_within_range(rule_data: dict, applicant_data: dict):
    # return:: True--> if condition match
    try:
        requested_loan_amount = int(applicant_data.get('requestedLoanAmount', 0))
        min_loan_amount = int(rule_data.get('MinLoan', 0))
        max_loan_amount = int(rule_data.get('MaxLoan', 0))

        if min_loan_amount <= requested_loan_amount <= max_loan_amount:
            return True
        else:
            return False
    except KeyError as e:
        return False
    
# Rule 3: Min Max Loan Term
def is_loan_term_within_range(rule_data: dict, applicant_data: dict):
    # return:: True--> if loan term within range
    try:
        loan_term = applicant_data['requestedLoanTerm']
        min_term = rule_data.get('MinTerm', 0)
        max_term = rule_data.get('MaxTerm', 0)
        if min_term <= loan_term <= max_term:
            return True
        return False
    except KeyError as e:
        return False

# Rule 4: Asset Condition: NEW=1, USED=2, PRE_APPROVAL=3
def check_asset_condition(rule_data: dict, applicant_data: dict) -> bool:
    # return:: True--> if assetCondition == assetCondition
    try:
        asset_condition = rule_data.get('AssetCondition', 'BOTH').upper()  # NEW, USED, BOTH
        asset_loan_type = applicant_data.get('assetCondition', '').upper()  # NEW, USED, PRE_APPROVAL

        # BOTH = NEW+USED
        if asset_loan_type == 'PRE_APPROVAL' or asset_condition == 'BOTH':
            return True
        else:
            return asset_loan_type == asset_condition
    except KeyError as e:
        return False

# Rule 5: Determine Car Age (Asset Age)
def get_asset_age(applicant_data: dict):
    # calculate vehicle_age = current year - asset_year
    current_year = int(date.today().strftime("%Y"))
    try:
        asset_condition = applicant_data.get('assetCondition', '').upper()
        if asset_condition == "NEW":
            return 0
        else:   # USED, BOTH
            return current_year - applicant_data['assetYear']
    except KeyError as e:
        print(f"KeyError: get_asset_age: {e}")
        return False

# Rule 6: Asset age (NEW [asset_condition='NEW'], USED [0-5years, 10+ years])   
def is_asset_age_within_range(rules_data: dict, applicant_data: dict) -> bool:
    # return:: True--> if condition match (NEW, USED, BOTH)
    # N/A = 0-99 [Min-Max Asset Age]
    try:
        asset_condition = applicant_data.get('assetCondition', '').upper()  # NEW, USED, BOTH
        assetYear = applicant_data.get('assetYear', 0)
        if assetYear <= 1980:
            return False
        if asset_condition == "NEW":
            return True
        else:   # USED, BOTH
            asset_age = get_asset_age(applicant_data=applicant_data)
            min_asset_age = rules_data.get('MinAssetAge', 0)
            max_asset_age = rules_data.get('MaxAssetAge', 0)
            return min_asset_age <= asset_age <= max_asset_age
    except KeyError as e:
        print(f"KeyError {rules_data.get('ProductId')}: MinAssetAge: {e}")
        return False
    except Exception as e:
        print(f"KeyError {rules_data.get('ProductId')}: MinAssetAge: {e}")
        return False

# Rule 7: Asset backed: if not homeowner return:: false    
def is_asset_backed_requirement_satisfy(rules_data: dict, applicant_data: dict):
    # return:: false, if not homeowner/Mortgage
    asset_backed_req = rules_data.get('isAssetBacked', False)
    current_residence_status = applicant_data.get('currentResidenceStatus', 'REAL_STATE_RENT').upper()
    is_home_owner = applicant_data.get('isHomeOwner', False)

    if asset_backed_req:
        # Home owner / Mortgage
        if current_residence_status in ["OWNED", "MORTGAGED"]:
            return True
        elif is_home_owner:
            return True
    else:
        # Asset backed not required
        return True

# Rule 8: Applicant Age 
def is_applicant_age_within_range(rules_data: dict, applicant_data: dict) -> bool:
    # N/A --> 16-99 Years
    try:
        min_age = rules_data.get('ApplicantMinAge', 0)
        max_age = rules_data.get('ApplicantMaxAge', 0)
        applicant_age = applicant_data.get('applicantAge', 0)
        if applicant_age is None:
            raise ValueError("Applicant age is missing from applicant data.")
        elif max_age in {'N/A', 'NOT_APPLICABLE'}:
            max_age = 99
        return min_age <= applicant_age <= max_age
    except (KeyError, ValueError, TypeError, Exception) as e:
        print(f"KeyError {rules_data.get('ProductId')}: applicantAge: {e}")
        return False        

# Rule 9: Acceptable Visa Type: [CITIZEN, PR, STUDENT, WORK, TRAVEL, OTHERS]    
def is_visa_type_accepted(rules_data: dict, applicant_data: dict):
    """
    return:: false, if conditions are not met.
    VISA Type --> CITIZEN, PERMANENT_RESIDENT, STUDENT_VISA, WORKING_VISA, NON_WORKING_VISA, TRAVEL_VISA
    """
    NON_WORKING_VISA = {"STUDENT_VISA", "TRAVEL_VISA", "TOURIST_VISA", "OTHER"}
    ALL_VISA_TYPES = NON_WORKING_VISA.union({"WORKING_VISA"})
    try:
        not_acceptable_visa_type = rules_data.get('NotAcceptableVisaType', '').upper()
        applicant_visa_type = applicant_data.get('visaType', '').upper()

        # TODO 3 years min left of visa - must not effect working rights
        if applicant_visa_type in ['CITIZEN', 'PERMANENT_RESIDENT']:
            return True
        elif not_acceptable_visa_type in {'N/A', 'NOT_APPLICABLE', ''}:
            return True
        elif not_acceptable_visa_type == 'KO' and not (
                applicant_visa_type == 'CITIZEN' or applicant_visa_type == 'PERMANENT_RESIDENT'):
            return False
        elif not_acceptable_visa_type == 'ALL_VISA_TYPE':
            # Any kind of visa is not acceptable
            return applicant_visa_type not in ALL_VISA_TYPES
        elif not_acceptable_visa_type == 'NON_WORKING_VISA':
            # Only 'Non-Working' visa is not acceptable
            return applicant_visa_type not in NON_WORKING_VISA
        else:
            return not_acceptable_visa_type != applicant_visa_type
    except (KeyError, ValueError, TypeError, Exception) as e:
        print(f"KeyError {rules_data.get('ProductId')}: notAcceptableVisaType: {e}")
        return False

# Rule 10: Applicant's income check
def is_applicant_income_sufficient(rules_data: dict, applicant_data: dict) -> bool:
    # applicantTotalIncomeMonthly ==> refer to combined of all incomes
    # i.e. rental Income + centrelink Income + uncommitted Income + others income
    try:
        min_income_require = rules_data.get('ApplicantMinIncome', 0)  # Monthly

        # Find the applicant's highest income from zink2 application vs bank statement
        highest_income = applicant_data.get('applicantTotalIncomeMonthly', 0)
        is_statement_available = applicant_data.get('isBankStatementAvailable', False)

        if min_income_require in {None, '', 'N/A'}:
            return True
        else:
            return highest_income >= int(min_income_require)
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: applicantMinIncome: {e}")
        return False

# Rule 11: Current Residence Status: OWNED, REAL_STATE_RENT, BOARDED, PRIVATE_RENT
def is_residence_months_sufficient(rules_data: dict, applicant_data: dict):
    # return:: false, if conditions are not met.
    try:
        current_residence_status = applicant_data.get('currentResidenceStatus', 'RENTED').upper()
        living_current_residence = applicant_data.get('currentResidenceMonths', 0)
        min_residence_months = {
            "BOARDED": rules_data['MonthAtBoarded'],
            "REAL_STATE_RENT": rules_data['MonthAtRealStateRent'],
            "PRIVATE_RENT": rules_data['MonthAtPrivateRent'],
            "MORTGAGED": rules_data['MonthAtMortgage'],
            "OWNED": rules_data['MonthAtOwned'],
            "RENTED": rules_data['MonthAtRealStateRent'] # TODO Delete 'RENTED' once changes done in ZINK2
        }.get(current_residence_status)

        # check the type and then convert to uppercase
        if isinstance(min_residence_months, str):
            min_residence_months = min_residence_months.upper()

        if min_residence_months in {None, '', 'N/A', 'Y', 'YES'}:
            return True
        elif min_residence_months in {'N', 'NO'}:
            # N = NO; Y = YES
            return False
        elif min_residence_months == 'KO':
            if living_current_residence > 0:
                return False
            else:
                return True
        elif int(living_current_residence) >= int(min_residence_months):
            return True
        else:
            return False
    except (KeyError, ValueError, TypeError, Exception) as e:
        print(f"KeyError {rules_data.get('ProductId')}: currentResidenceStatus: {e}")
        return False

# Rule 12: Check Applicant Credit Score
def check_applicant_credit_score(rules_data: dict, applicant_data: dict):
    try:
        credit_ver = float(rules_data.get('Version', 2))

        applicant_credit_score = 0
        if credit_ver == 1.0:
            applicant_credit_score = int(applicant_data.get('creditScoreV10', 0))
        elif credit_ver == 1.1:
            applicant_credit_score = int(applicant_data.get('creditScoreV11', 0))
        elif credit_ver == 2.0:
            applicant_credit_score = int(applicant_data.get('creditScoreV20', 0))
        return applicant_credit_score
    except (ValueError, TypeError, KeyError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: creditReportVersion: {e}")
        return False

# Rule 13: Special score values: -900 series (For bankrupt applicant)
def is_900_series_allowed(rules_data: dict) -> bool:
    # Special score values: -900 series
    ALLOWED_SPECIAL_SCORES = {-901, -902, -903, -993, -994, -995, -996, -997, -998, -999}
    try:
        allowed_codes = rules_data.get('Allowed900SeriesCode', 'N/A')

        if allowed_codes.upper() == "ALL":
            allowed_codes = ALLOWED_SPECIAL_SCORES

        # splitting the string, and then converting the list to a set using a set comprehension.
        if allowed_codes is not None and allowed_codes.strip() != "N/A" and allowed_codes.strip() != '':
            lender_allowed_codes = {int(x.strip()) for x in allowed_codes.split(",")}  # -996

            if lender_allowed_codes is not None:
                if lender_allowed_codes.issubset(ALLOWED_SPECIAL_SCORES):
                    return True
        return False
    except (ValueError, TypeError, KeyError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: Allowed900SeriesCode: {e}")
        return False

# Rule 14: Check Min Credit Score
def meet_credit_score_eligibility(rules_data: dict, applicant_data: dict) -> bool:
    try:
        # Applicant credit score by credit version in rules
        applicant_credit_score = check_applicant_credit_score(rules_data, applicant_data)

        credit_score_required = rules_data.get('CreditScore', 0)
        allowed_codes = rules_data.get('Allowed900SeriesCode', 'N/A')

        if credit_score_required in {None, '', 'N/A'}:
            return True
        elif int(credit_score_required) == 0:
            return True
        elif applicant_credit_score < 0:  # For Negative credit score (bankrupt)
            if allowed_codes is not None and allowed_codes.strip() != "N/A":
                return bankrupt_rules.is_900_series_allowed(rules_data)
            else:
                return True
        else:
            # credit score > 0
            return applicant_credit_score >= int(credit_score_required)
    except (ValueError, TypeError, KeyError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: CreditScore: {e}")
        return False

# Rule 15: checks if the count of current defaults of the applicant exceed the lender's allowable current defaults.
def check_current_default_allow(rules_data: dict, applicant_data: dict) -> bool:
    try:
        countCurrentDefault = applicant_data.get('countTotalCurrentDefault', 0)
        current_default_allow = rules_data.get('CurrentDefaultAllow', "N/A")  # Financial + Non-Financial

        if isinstance(current_default_allow, str):
            current_default_allow = current_default_allow.upper()

        # current_default_allow = '1 in 12 months=ACCEPT'
        current_default_allow = current_default_allow.replace(" ", "")  # Remove all spaces
        matches = re.findall(r'(\d+)in(\d+)months=([A-Z]+)', current_default_allow, re.IGNORECASE)  # 1 in 12 months

        if current_default_allow in {None, '', 'N/A'}:
            return True
        elif current_default_allow == 'KO':
            return countCurrentDefault == 0

        # [ 1 in 12 months=ACCEPT ] OR [ 1 in 12 months=KO ]
        elif matches:
            default_allow, max_month_allow, action = matches[0]

            if action == 'ACCEPT':  # 1 in 12 months=ACCEPT
                applicant_default = 0
                if int(max_month_allow) == 12:
                    applicant_default = applicant_data.get('countTotalCurrentDefault12Month', 0)
                return applicant_default <= int(default_allow)

            elif action == 'KO':  # 1 in 12 months=KO
                if int(max_month_allow) == 12:
                    applicant_default = applicant_data.get('countTotalCurrentDefault12Month', 0)
                    return int(default_allow) > applicant_default  # or False
        else:
            return int(countCurrentDefault) <= int(current_default_allow)
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: currentDefaultAllow: {e}")
        return False

# Rule 16: Check total UNPAID financial defaults
def unpaid_financial_default(rules_data: dict, applicant_data: dict):
    """
        Possible values: KO, N/A, $4000, 5
    """
    try:
        unpaidFinancialDefaultAmount = applicant_data.get('totalFinancialUnPaidDefaultAmount', 0)
        unpaidFinancialDefaultCount = applicant_data.get('countFinancialUnPaidDefault', 0)
        maxUnpaidFinancialDefaultAllow = rules_data.get('CountUnpaidFinancialDefaultAllow', 0)  # acceptable default count

        if isinstance(maxUnpaidFinancialDefaultAllow, str):
            maxUnpaidFinancialDefaultAllow = maxUnpaidFinancialDefaultAllow.upper()

        if maxUnpaidFinancialDefaultAllow in {None, '', 'N/A'}:
            return True
        elif maxUnpaidFinancialDefaultAllow == 'KO':
            if unpaidFinancialDefaultCount == 0 and unpaidFinancialDefaultAmount == 0:
                return True
            else:
                return False
        else:
            # Number of unpaid financial defaults
            return int(unpaidFinancialDefaultCount) <= int(maxUnpaidFinancialDefaultAllow)
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: MaxUnpaidFinancialDefaultAllow: {e}")
        return False

# Rule 17: Current Worst Repayment Status Last 24 Month - Any Account
def is_24month_repayment_status_accepted_any_account(rules_data: dict, applicant_data: dict):
    try:
        worstRepaymentStatusAccepted = rules_data.get('WorstRepayment24MonthAnyAcc', '0')
        countWorstRepayment = applicant_data.get('countWorstRepayment24MonthAny', 0)
        if countWorstRepayment == 'X':  # X means consumer credit is 180 + days overdue
            countWorstRepayment = 7
        elif countWorstRepayment == '-991':
            countWorstRepayment = 0

        if isinstance(worstRepaymentStatusAccepted, str):
            worstRepaymentStatusAccepted = worstRepaymentStatusAccepted.upper()

        # Worst repayment status = '1 in 24 months=ACCEPT'
        worst_repayment_status = worstRepaymentStatusAccepted.replace(" ", "")  # Remove all spaces
        matches = re.findall(r'(\d+)in(\d+)months=([A-Z]+)', worst_repayment_status, re.IGNORECASE)  # 1 in 24 months

        if worstRepaymentStatusAccepted in {None, '', 'N/A'}:
            return True, ''
        elif worstRepaymentStatusAccepted == 'KO':
            if int(countWorstRepayment) <= 0:
                return True, ''
            else:
                return False, ''
        else:
            return int(countWorstRepayment) <= int(worstRepaymentStatusAccepted), ''
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: WorstRepayment24MonthAnyAcc: {e}")
        return False

# Rule 18: Current Worst Repayment Status Last 24 Month - Auto
def is_24month_repayment_status_accepted_Auto(rules_data: dict, applicant_data: dict):
    try:
        worstRepaymentStatusAccepted = rules_data.get('WorstRepayment24MonthAuto', '0')
        countWorstRepayment = applicant_data.get('countWorstRepayment24MonthAuto', 0)
        if countWorstRepayment == 'X':  # X means consumer credit is 180 + days overdue
            countWorstRepayment = 7
        elif countWorstRepayment == '-991':
            countWorstRepayment = 0

        if isinstance(worstRepaymentStatusAccepted, str):
            worstRepaymentStatusAccepted = worstRepaymentStatusAccepted.upper()

        # Worst repayment status = '1 in 24 months=ACCEPT'
        worst_repayment_status = worstRepaymentStatusAccepted.replace(" ", "")  # Remove all spaces
        matches = re.findall(r'(\d+)in(\d+)months=([A-Z]+)', worst_repayment_status, re.IGNORECASE)  # 1 in 24 months

        if worstRepaymentStatusAccepted in {None, '', 'N/A'}:
            return True, ''
        elif worstRepaymentStatusAccepted == 'KO':
            if int(countWorstRepayment) <= 0:
                return True, ''
            else:
                return False, ''
        else:
            return int(countWorstRepayment) <= int(worstRepaymentStatusAccepted), ''
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: WorstRepayment24MonthAuto: {e}")
        return False       

# Rule 19: Max Dishonor: Any payment failed 
def is_financial_dishonor_acceptable(rules_data: dict, applicant_data: dict) -> bool:
    try:
        totalDishonor = applicant_data.get('countDishonor', 0)
        maxFinancialDishonorAllow = rules_data.get('maxFinancialDishonorAllow', 0)
        if maxFinancialDishonorAllow is None or maxFinancialDishonorAllow == '' or maxFinancialDishonorAllow == 'N/A':
            return True
        return totalDishonor <= maxFinancialDishonorAllow
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: maxFinancialDishonorAllow: {e}")
        return False

# Rule 20: Max ongoing active loan Amount: currently paying   
def ongoing_payday_loan_eligibility(rules_data: dict, applicant_data: dict) -> bool:
    try:
        ongoing_payday_loan_amount = applicant_data.get('ongoingSaccRepaymentAmount', 0)
        maxOngoingPaydayLoanAllow = rules_data.get('MaxOngoingPaydayLoan', '0')

        if isinstance(maxOngoingPaydayLoanAllow, str):
            maxOngoingPaydayLoanAllow = maxOngoingPaydayLoanAllow.upper()

        if maxOngoingPaydayLoanAllow in {None, '', 'N/A'}:
            return True
        elif maxOngoingPaydayLoanAllow == 'KO':
            if ongoing_payday_loan_amount == 0:
                return True
            else:
                return False
        else:
            return int(ongoing_payday_loan_amount) <= int(maxOngoingPaydayLoanAllow)
    except (KeyError, TypeError, ValueError) as e:
        print(f"KeyError {rules_data.get('ProductId')}: MaxOngoingPaydayLoan: {e}")
        return False


In [6]:
# Rule 21: Current Employment duration [UNEMPLOYED, RETIRED, PENSIONER, CASUAL, PART_TIME, FULL_TIME, OTHERS]
def is_employment_duration_sufficient(rules_data: dict, applicant_data: dict):
    """
        return:: false, if conditions are not met.
        Employment Values: 12, KO, N, Y, N/A, CUSTOM EQUATION
    """
    current_emp_type = applicant_data.get('employmentType', 'UNEMPLOYED').upper()
    is_probation_allow = rules_data.get('isProbationAllow', False)
    is_applicant_probation = applicant_data.get('isProbation', False)

    # Calculate total employment duration (Previous and Current)
    current_emp_duration = applicant_data.get('employmentDurationMonths', 0)
    all_previous_emp_duration = get_previous_employment_duration(applicant_data=applicant_data)
    total_employment_duration = current_emp_duration + all_previous_emp_duration

    # If NO employment, Consider Centrelink Income
    if current_emp_type in {None, '', 'N/A', 'UNEMPLOYED'}:
        centrelink_income = applicant_data.get('centrelinkIncomeMonthly', 0)
        min_income_required = rules_data.get('applicantMinIncome', 0)
        return centrelink_income >= min_income_required, config.CENTRELINK_INCOME

    # Is Probation Allowed?
    elif is_applicant_probation and is_probation_allow:
        return True, 0
    else:
        # get employment duration required by current employment type
        month_required_rule = get_required_employment_duration(rules_data, applicant_data)  # type=string

        if isinstance(month_required_rule, str):
            month_required_rule = month_required_rule.upper()

        try:
            if month_required_rule in {'Y', 'YES', 'N/A', ''}:
                return True, 0
            elif month_required_rule in {'N', 'NO'}:
                return True, config.SOFT_NO
            elif month_required_rule == 'KO':
                return False, month_required_rule   # return KO
            elif isinstance(month_required_rule, int):
                return int(total_employment_duration) >= int(month_required_rule), 0

            # Evaluate Rules
            elif month_required_rule.startswith('EMPLOYMENT=ACCEPTED') and 'IF' in month_required_rule:
                is_continuation_allow = rules_data.get('isContinuationAllow', False)

                product_rules = month_required_rule.split(';')
                employment_parts = product_rules[0]  # EMPLOYMENT=ACCEPTED
                condition_parts = product_rules[1]  # IF(CONTINUATION>=6&CURRENT_EMP_DURATION>=2)
                conditions_list = condition_parts.split('(')[1].split(')')[0].split('&')

                for condition in conditions_list:
                    # Rule 1: Asset Backed Required (HomeOwner)
                    # EMPLOYMENT=ACCEPTED;IF(ASSET_TYPE==1&CURRENT_EMP_DURATION>=12)
                    # ASSET_TYPE==1 --> Homeowner;  ASSET_TYPE==0 --> Non Homeowner
                    if "ASSET_TYPE" in condition:
                        valid_conditions = all(eval(condition.replace('CURRENT_EMP_DURATION',str(applicant_data['employmentDurationMonths']))
                                                    .replace('ASSET_TYPE',str(applicant_data.get('isHomeOwner', 0))))
                                               for condition in conditions_list)
                        if valid_conditions:
                            return True, 0
                        return False, config.HARD_KO  # Asset Backed Required

                    # Rule 2: Current Employment + Continuation Allow
                    # EMPLOYMENT=ACCEPTED;IF(CONTINUATION>=6&CURRENT_EMP_DURATION>=1)
                    elif is_continuation_allow and "CONTINUATION" in condition:
                        # If NO previous employment history
                        if all_previous_emp_duration == 0:
                            pattern = r'[\d\.]+'  # pattern to extract values
                            emp_duration_required = 0
                            for string in conditions_list:
                                match = re.search(pattern, string)
                                if match:
                                    emp_duration_required += (int(match.group()))
                            return current_emp_duration >= emp_duration_required, current_emp_duration

                        # If previous employment history available
                        else:
                            valid_conditions = all(eval(condition.replace('CURRENT_EMP_DURATION', str(applicant_data['employmentDurationMonths']))
                                                        .replace('CONTINUATION', str(all_previous_emp_duration)))
                                                   for condition in conditions_list)
                            if valid_conditions:
                                return True, 0
                            return False, config.SOFT_KO
                    else:
                        return False, config.CONTINUATION_NOT_ALLOW  # return KO
            elif int(total_employment_duration) >= int(month_required_rule):
                return True, month_required_rule
        except Exception as e:
            print(f"KeyError {rules_data.get('ruleId')}: is_employment_duration_sufficient: {e}")
            return False, 0

def get_previous_employment_duration(applicant_data: dict):
    """
    Calculate sum of previous employment duration
    """
    current_emp_type = applicant_data.get('employmentType', 'UNEMPLOYED').upper()
    previous_ft_duration = 0
    previous_pt_duration = 0
    previous_casual_duration = 0
    previous_pensioner_duration = 0
    previous_unemployed_duration = 0

    for key, emp_type in applicant_data.items():
        if isinstance(emp_type, str):
            emp_type = emp_type.upper()

        if "previousEmpType" in key:
            type_num = key[-1]  # Extract the number from the key
            duration_key = "previousEmpDurationMonths" + str(type_num)
            if emp_type in {'FULL_TIME', 'SELF_EMPLOYED', 'CONTRACT'}:
                previous_ft_duration += applicant_data.get(duration_key, 0)
            elif emp_type == "PART_TIME":
                previous_pt_duration += applicant_data.get(duration_key, 0)
            elif emp_type == "CASUAL":
                previous_casual_duration += applicant_data.get(duration_key, 0)
            elif emp_type == "PENSIONER":
                previous_pensioner_duration += applicant_data.get(duration_key, 0)
            elif emp_type == "UNEMPLOYED":
                previous_unemployed_duration += applicant_data.get(duration_key, 0)

    # Return total previous employment duration based on the current employment type
    if current_emp_type in {'FULL_TIME', 'SELF_EMPLOYED', 'CONTRACT'}:
        previous_ft_duration = previous_ft_duration + (previous_pt_duration/2) + (previous_casual_duration/4)
        return previous_ft_duration
    elif current_emp_type in {'PART_TIME'}:
        previous_pt_duration = (previous_ft_duration * 1) + previous_pt_duration + (previous_casual_duration/4)
        return previous_pt_duration
    elif current_emp_type == "CASUAL":
        previous_casual_duration = (previous_ft_duration * 1) + previous_pt_duration + previous_casual_duration
        return previous_casual_duration
    elif current_emp_type == "PENSIONER":
        return previous_pensioner_duration
    elif current_emp_type == "UNEMPLOYED":
        # previous_unemployed_duration = (previous_ft_duration * 1) + previous_pt_duration + + (previous_casual_duration/4)
        return previous_unemployed_duration
    else:
        return 0

def get_required_employment_duration(rule_data: dict, applicant_data: dict):
    """
        Calculate employment duration required by current employment type
    """
    current_emp_type = applicant_data.get('employmentType', 'UNEMPLOYED').upper()

    employment_type_keys = {
        "UNEMPLOYED": rule_data.get("MinMonthUnemployed", '0'),
        "STUDENT": rule_data.get("MinMonthStudent", '0'),
        "RETIRED": rule_data.get("MinMonthRetired", '0'),
        "PENSIONER": rule_data.get("MinMonthPensioner", '0'),
        "CASUAL": rule_data.get("MinMonthCasual", '0'),
        "CONTRACT": rule_data.get("MinMonthFullTime", '0'),
        "PART_TIME": rule_data.get("MinMonthPartTime", '0'),
        "FULL_TIME": rule_data.get("MinMonthFullTime", '0'),
        "SELF_EMPLOYED": rule_data.get("MinMonthAbn", '0'),  # Commercial required ABN Registration
        "OTHER": rule_data.get("MinMonthOthers", '0')
    }
    month_required_rule = employment_type_keys.get(current_emp_type)  # type=string
    return month_required_rule


In [7]:
# Determine Product Knock Out (KO) Reason
def products_ko_block(ko_list: list, matrix_rule_data: dict):
    ko_dict = {
        "lender_id": matrix_rule_data.get("LenderId", "Undefined"),
        "product_id": matrix_rule_data.get("ProductId", "Undefined"),
        "product_name": matrix_rule_data.get('ProductName', "Undefined"),
        "ko_notes": ko_list
    }
    return ko_dict

def products_qualified_block(matrix_rule_data: dict):
    repayment_amount = 0
    
    product_option_block = {
        "lender_id": matrix_rule_data.get("LenderId", "Undefined"),
        "product_id": matrix_rule_data.get("ProductId", "Undefined"),
        "product_name": matrix_rule_data.get('ProductName', "Undefined"),
        "min_loan_term": matrix_rule_data['MinTerm'],
        "max_loan_term": matrix_rule_data['MaxTerm'],
        "max_loan_amount": matrix_rule_data['MaxLoan'],
        "min_interest_rate": matrix_rule_data['BaseRate'],
        "max_interest_rate": matrix_rule_data['MaxRate'],
        "min_repayment_monthly": round(repayment_amount, 2)
    }
    return product_option_block

In [8]:
# ****************************
#    Rule Assessmement 
# ****************************
ko_list = []
qualified_product_list = []
product_ko_list = []

for index, applicant_data in df_loan_app.iterrows():
    # Convert applicant dataframe to dict
    applicant_dict = applicant_data.to_dict()

    for index, matrix_data in df_product_matrix.iterrows():
        # Convert product matrix dataframe to dict
        rule_data = matrix_data.to_dict()
        rule_id = int(rule_data.get('ProductId', 0))
        # Clear ko_list for each product evaluation
        ko_list = []
        
        # [1] Check whether product is enabled
        if not is_product_enabled(rule_data):
            ko_list.append(f'{rule_id}: Product is DISABLE. Please contact admin.')

        # [2] Min Max Loan Amount
        if not is_loan_amount_within_range(rule_data, applicant_data):
            max_loan_amount_allowed = int(rule_data.get('MaxLoan', 0))
            ko_list.append(f"Max Loan amount should be < ${str(max_loan_amount_allowed)}.")

        # [3] Min Max Loan Term
        if not is_loan_term_within_range(rule_data, applicant_dict):
            ko_list.append(f'{rule_id}: Loan term EXCEED.')

        # [4] Asset Condition: NEW=1, USED=2, PRE_APPROVAL=3
        if not check_asset_condition(rule_data, applicant_data):
            asset_condition = rule_data.get('AssetCondition', '').upper()
            ko_list.append(f"Asset condition '{asset_condition}' is not match.")

        # [6] Asset age (NEW [asset_condition='NEW'], USED [0-5years, 10+ years])
        if not is_asset_age_within_range(rule_data, applicant_data):
            min_asset_age = rule_data.get('MinAssetAge', 0)
            max_asset_age = rule_data.get('MaxAssetAge', 0)
            ko_list.append(f"Asset age is not within expected range ({min_asset_age} - {max_asset_age}) years.")

        # [7] Asset backed: is applicant homeowner
        if not is_asset_backed_requirement_satisfy(rule_data, applicant_data):
            ko_list.append('Asset backed is required (Home owner/Mortgage).')

        # [8] Applicant Age 
        if not is_applicant_age_within_range(rule_data, applicant_data):
            applicant_age = applicant_data.get('applicantAge', 0)
            ko_list.append(f"Applicant age is {str(applicant_age)} which does not meet lending criteria.")

        # [9] Acceptable Visa Type: [CITIZEN, PR, STUDENT, WORK, TRAVEL, OTHERS]
        if not is_visa_type_accepted(rule_data, applicant_data):
            visa_type = applicant_data.get('visaType', '').upper()
            ko_list.append(f"Visa type '{visa_type}' is not acceptable.")

        # [10] Applicant's income check
        if not is_applicant_income_sufficient(rule_data, applicant_data):
            applicant_income = round(applicant_data.get('totalIncomeMonthly', 0), 2)
            min_income_req = int(rule_data.get('ApplicantMinIncome', 0))  # Monthly
            ko_list.append(f"Applicant income ${str(applicant_income)} < ${str(min_income_req)}")

        # [11] Current Residence Status: OWNED, REAL_STATE_RENT, BOARDED, PRIVATE_RENT
        if not is_residence_months_sufficient(rule_data, applicant_data):
            residence_status = applicant_data.get('currentResidenceStatus', 0)
            ko_list.append(f"Living arrangement through '{residence_status}' cannot be considered.")

        # [14] Check Min Credit Score including special scores (-900 series)
        if not meet_credit_score_eligibility(rule_data, applicant_data):
            score_required = rule_data.get('CreditScore', 0)
            ko_list.append(f"Credit score {str(score_required)} is required.")

        # [15] checks if the count of current defaults
        if not check_current_default_allow(rule_data, applicant_data):
            current_default_allow = rule_data.get('CurrentDefaultAllow', 0)
            count_current_default = applicant_data.get('countTotalCurrentDefault', 0)
            ko_list.append(f"Number of Current Defaults {str(count_current_default)} > {str(current_default_allow)}")

        # [16] Check total UNPAID financial defaults
        if not unpaid_financial_default(rule_data, applicant_data):
            maxUnpaidFinancialDefaultAllow = rule_data.get('CountUnpaidFinancialDefaultAllow', 0)
            countUnpaidFinancialDefault = applicant_data.get('countFinancialUnPaidDefault', 0)
            ko_list.append(f"Total Unpaid Financial Defaults {str(countUnpaidFinancialDefault)} > {str(maxUnpaidFinancialDefaultAllow)}")

        # [17] Current Worst Repayment Status Last 24 Month - Any Account
        if not is_24month_repayment_status_accepted_any_account(rule_data, applicant_data):
            countWorstRepaymentAny24m = applicant_data.get('countWorstRepayment24MonthAny', 0)
            worstRepaymentAnyAcc24m = rule_data.get('WorstRepayment24MonthAnyAcc', 0)
            ko_list.append(f"Worst any account repayment CCR status last 24m {countWorstRepaymentAny24m} > {str(worstRepaymentAnyAcc24m)}")

        # [18] Current Worst Repayment Status Last 24 Month - Auto
        if not is_24month_repayment_status_accepted_Auto(rule_data, applicant_data):
            countWorstRepaymentAuto24m = applicant_data.get('countWorstRepayment24MonthAuto', 0)
            worstRepaymentAuto24m = rule_data.get('WorstRepayment24MonthAuto', 0)
            ko_list.append(f"Worst auto loan repayment CCR status last 24m {countWorstRepaymentAuto24m} > {str(worstRepaymentAuto24m)}")

        # [19] Max Dishonor: Any payment failed 
        if not is_financial_dishonor_acceptable(rule_data, applicant_data):
            totalDishonor = applicant_data.get('countDishonor', 0)
            maxFinancialDishonorAllow = rule_data.get('MaxFinancialDishonorAllow', 0)
            ko_list.append(f"Number of Dishonor {totalDishonor} > {str(maxFinancialDishonorAllow)}")

        # [20] Max ongoing active loan Amount: currently paying   
        if not ongoing_payday_loan_eligibility(rule_data, applicant_data):
            ongoingPaydayAmount = applicant_data.get('maxOngoingPaydayLoanAmount', 0)
            maxOngoingPaydayLoan = rule_data.get('MaxOngoingPaydayLoan', 0)
            ko_list.append(f"Total ongoing payday loan amount {str(ongoingPaydayAmount)}> {str(maxOngoingPaydayLoan)}")

        # [21] Current Employment duration
        if not is_employment_duration_sufficient(rule_data, applicant_data):
            employment_type = applicant_data.get('employmentType', 'UNEMPLOYED').upper()
            ko_list.append(f"{employment_type} employment is not acceptable.")

        if ko_list:
            # Capture Product Knock Out (KO) Reason
            product_ko = products_ko_block(ko_list=ko_list, matrix_rule_data=rule_data)
            product_ko_list.append(product_ko)
        else:
            # Qualified Product
            qualified_product = products_qualified_block(matrix_rule_data=rule_data)
            qualified_product_list.append(qualified_product)


In [9]:
print(len(qualified_product_list))

2


In [10]:
qualified_product_list

[{'lender_id': 160,
  'product_id': 9417,
  'product_name': 'Metro Finance - Consumer - Tier 2 - Asset Backed',
  'min_loan_term': 12,
  'max_loan_term': 84,
  'max_loan_amount': 250000,
  'min_interest_rate': 9.99,
  'max_interest_rate': 9.99,
  'min_repayment_monthly': 0},
 {'lender_id': 160,
  'product_id': 9420,
  'product_name': 'Metro Finance - Consumer - Tier 3 - Asset Backed',
  'min_loan_term': 12,
  'max_loan_term': 84,
  'max_loan_amount': 50000,
  'min_interest_rate': 11.39,
  'max_interest_rate': 11.39,
  'min_repayment_monthly': 0}]